# Importation et embeddings des données

In [ ]:
import pandas as pd
songs = pd.read_csv("all_songs.csv")
songs = songs.dropna(subset=["paroles"]).copy()
songs = songs[songs["paroles"].str.strip() != ""].reset_index(drop=True)

In [7]:
import numpy as np
from transformers import GPT2Tokenizer, GPT2Model
tokenizer = GPT2Tokenizer.from_pretrained('lucone83/deep-metal')
model = GPT2Model.from_pretrained('lucone83/deep-metal')
import torch
def embed_texte(texte):
    inputs = tokenizer(texte, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = model(**inputs)
    return output.last_hidden_state.mean(dim=1).squeeze(0).numpy()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 331.43it/s]
[transformers] GPT2Model LOAD REPORT from: lucone83/deep-metal
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from tqdm.auto import tqdm
vecteurs = np.vstack([embed_texte(t) for t in tqdm(songs["paroles"], desc="Embeddings")])

Embeddings: 100%|██████████| 32007/32007 [2:18:10<00:00,  3.86it/s]  


In [9]:
emb = pd.DataFrame(
     vecteurs,
     index=pd.MultiIndex.from_frame(songs[["genre", "artiste", "titre", "position"]]),
     columns=[f"dim_{i}" for i in range(vecteurs.shape[1])]
 )

In [10]:
emb.to_csv("embeddings_par_genre_all.csv")

## Classification

In [11]:
import pandas as pd
emb = pd.read_csv("embeddings_par_genre_all.csv", index_col=["genre", "artiste", "titre", "position"])